# P7 - Pousser des donnees, reentrainer, promouvoir en blue/green

Notebook de demonstration, execute **dans Fabric**. Il deroule la boucle MLOps complete :

| Etape | Ce qui se passe | Duree |
|---|---|---|
| 0 | Pousser des donnees dans un **nouveau regime** via l'Eventstream | 2 min |
| 1 | Snapshot des features -> datastore AML -> `telemetry_features:N+1` | 3 min |
| 2 | Pipeline `prep -> train -> evaluate -> register` -> nouveau modele | ~10 min |
| 3 | Deploiement `green` a 0%, test a chaud, puis **30%** | 8 a 15 min |
| 4 | Promotion a **100%**, `blue` conserve pour le rollback | 1 min |

> **Le message de la demo** : entre l'etape 3 et l'etape 4, la Function n'est **pas**
> redeployee. Meme URI de scoring, meme jeton, meme code. Les colonnes `deployment` et
> `model_version` de `scoring_results` changent toutes seules, parce que la Function lit
> le deploiement dans l'en-tete de reponse et resout la version aupres d'ARM.

## Prerequis

- `scripts/42_fabric_p7.ps1` a ete execute : il publie ce notebook **et** televerse le
  dossier `aml/` du depot vers le stockage. Sans lui, l'etape 2 ne trouve ni le pipeline
  ni ses composants - un notebook Fabric n'a pas le depot sous la main.
- La capacite Fabric tourne, l'Eventstream est actif, la Function tourne.
- Vous avez *Storage Blob Data Contributor* sur le compte de stockage et *Contributor*
  sur le workspace AML.

## Ce qui n'est pas automatise ici, et pourquoi

Le simulateur est un processus **local** qui poste vers l'endpoint personnalise de
l'Eventstream. Cet endpoint s'authentifie par chaine de connexion SAS : l'appeler depuis
Fabric supposerait de faire transiter une cle par ce notebook, ce que la solution
s'interdit partout ailleurs. L'etape 0 vous donne donc la commande a lancer en local,
puis **verifie** que les donnees sont bien arrivees par le vrai chemin de production.

Reimplanter la physique du simulateur ici aurait ete l'autre option : elle aurait cree
un second generateur a maintenir, pour eviter une seule commande.

In [ ]:
# Fabric requires %pip install to be in the FIRST cell of the notebook: it is a
# session-level operation, and running it later restarts the Python interpreter
# and discards every variable defined before it.
%pip install azure-ai-ml azure-kusto-data --quiet

## 0. Pousser des donnees dans un nouveau regime

Relancez le simulateur **en local** avec le regime modifie :

```
python simulator/sensor_producer.py --thermal-gain 40 --ambient-offset 5 --backfill-minutes 240
```

Deux effets distincts, volontairement combines :

- `--thermal-gain 40` **casse la relation** charge -> temperature que le modele a apprise
  (elle valait 25). L'erreur observee monte : c'est la derive de *performance*.
- `--ambient-offset 5` deplace la distribution des entrees elles-memes : c'est la derive
  de *donnees*, que l'on detecte sans connaitre la verite terrain.

`--backfill-minutes 240` rejoue quatre heures d'historique instantanement, puis continue
en temps reel. Sans lui il faudrait attendre des heures avant d'avoir un dataset
exploitable - le rythme d'une demo ne le permet pas.

**Laissez le simulateur tourner** pendant toute la demo : la Function ignore les features
de plus de 180 s, donc si le flux s'arrete, le scoring s'arrete aussi et les etapes 3 et 4
n'auront rien a montrer.

In [ ]:
# --- Identities and names -------------------------------------------------
SUBSCRIPTION_ID = '<SUBSCRIPTION_ID>'
RESOURCE_GROUP = 'rg-rti-aml-demo'
AML_WORKSPACE = 'aml-rti-demo'

WORKSPACE_ID = '<FABRIC_WORKSPACE_ID>'
KQL_DATABASE_ID = '<KQL_DATABASE_ID>'
KUSTO_URI = 'https://<EVENTHOUSE_HOST>.z7.kusto.fabric.microsoft.com'
KUSTO_DB = 'sensors'
TABLE_NAME = 'telemetry'

STORAGE_ACCOUNT = '<STORAGE_ACCOUNT>'
FILESYSTEM = 'snapshots'
DATASTORE = 'rti_snapshots'
PIPELINE_SOURCE_PREFIX = 'pipeline-source'

DATA_ASSET = 'telemetry_features'
MODEL_NAME = 'temp_forecaster'
ENDPOINT = 'temp-forecast-ep'
BLUE = 'blue'
GREEN = 'green'

# Traffic given to green at step 3, before the full promotion at step 4.
CANARY_PERCENT = 30

# --- Feature contract -----------------------------------------------------
# Must stay identical to fabric/kql/04_feature_function.kql, which is what the
# Function calls at serving time. Training/serving skew produces no error and a
# quietly worse model, which is the most expensive failure mode in this loop.
HORIZON_MIN = 15
WINDOW_MIN = 30
LAGS_MIN = [1, 5, 15]
SENSORS = ['temperature_c', 'humidity_pct', 'pressure_bar', 'vibration_mm_s']
ROLLING_SENSORS = ['temperature_c', 'vibration_mm_s']
TARGET = 'target_temperature_c'

# Hard floor per device, in usable minutes. Below this the model reports a
# respectable MAE and forecasts nothing.
MIN_ROWS_PER_DEVICE = 120

ONELAKE_PATH = ('abfss://' + WORKSPACE_ID + '@onelake.dfs.fabric.microsoft.com'
                + '/' + KQL_DATABASE_ID + '/Tables/' + TABLE_NAME)
ABFSS_ROOT = 'abfss://' + FILESYSTEM + '@' + STORAGE_ACCOUNT + '.dfs.core.windows.net'
MOUNT_POINT = '/snapshots'

print('source     :', ONELAKE_PATH)
print('destination:', ABFSS_ROOT)

In [ ]:
import base64
import json

import notebookutils
from azure.core.credentials import AccessToken


class FabricTokenCredential:
    """TokenCredential backed by the Fabric session identity.

    Fabric notebooks do NOT support DefaultAzureCredential: there is no managed
    identity endpoint, no CLI, no environment secret. The only token source is
    notebookutils, which is not a TokenCredential. This adapter is the bridge.

    getToken is picky about the audience string, and which forms it accepts has
    changed across runtimes. Rather than hardcode one and get an opaque failure
    mid-demo, we try the plausible forms and report which one worked.
    """

    def get_token(self, *scopes, **kwargs):
        scope = scopes[0] if scopes else 'https://management.azure.com/.default'
        base = scope.rsplit('/.default', 1)[0]
        candidates = [base, base + '/.default', scope]
        errors = []
        for audience in candidates:
            try:
                raw = notebookutils.credentials.getToken(audience)
            except Exception as exc:
                errors.append(str(audience) + ' -> ' + str(exc))
                continue
            payload = raw.split('.')[1]
            payload += '=' * (-len(payload) % 4)
            claims = json.loads(base64.urlsafe_b64decode(payload))
            return AccessToken(raw, int(claims['exp']))
        raise RuntimeError('No usable audience for scope ' + scope + ':\n  ' + '\n  '.join(errors))


cred = FabricTokenCredential()
print('ARM token acquired, expires at epoch', cred.get_token('https://management.azure.com/.default').expires_on)

In [ ]:
from azure.ai.ml import MLClient
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table

ml = MLClient(cred, SUBSCRIPTION_ID, RESOURCE_GROUP, AML_WORKSPACE)
kusto = KustoClient(KustoConnectionStringBuilder.with_azure_token_credential(KUSTO_URI, cred))


def kql(query):
    """Run KQL and return a pandas DataFrame.

    If this raises on the token audience, do not fight it during the demo: every
    query in this notebook is plain KQL and can be pasted into a queryset on the
    'sensors' database. Only the observation cells depend on it.
    """
    return dataframe_from_result_table(kusto.execute(KUSTO_DB, query).primary_results[0])


print('AML workspace   :', ml.workspace_name)
print('endpoint traffic:', ml.online_endpoints.get(ENDPOINT).traffic)
display(kql('telemetry | summarize rows = count(), newest = max(timestamp), lag = now() - max(timestamp)'))

In [ ]:
# Did the regime shift actually land? Compare the last 30 minutes against a
# window from before the restart. If these look alike, the simulator is still
# running in nominal mode - and retraining would produce a twin of blue, i.e. a
# demo with nothing to show. Better to find out now than after 25 minutes of
# training and deployment.
display(kql('''
telemetry
| where timestamp > ago(6h)
| summarize
    temperature_c  = avg(temperature_c),
    vibration_mm_s = avg(vibration_mm_s),
    samples        = count()
  by bin(timestamp, 30m)
| order by timestamp asc
'''))

## 1. Nouveau dataset

Le snapshot lit la table `telemetry` **mirroree dans OneLake**, pas l'Eventhouse en direct.
C'est le meme chemin que celui qui a produit la `v1`, ce qui garantit que le nouveau modele
est entraine sur des features construites exactement comme celles de son predecesseur. Une
comparaison blue/green n'a de sens que si la seule variable qui change est le modele.

**Le mirroring est asynchrone** (`TargetLatencyInMinutes = 5`). Les donnees que vous venez
de pousser mettent donc jusqu'a cinq minutes a apparaitre ici. La cellule suivante affiche
la borne haute reelle : si elle est trop ancienne, attendez une fenetre et relancez-la.

In [ ]:
def next_version(existing):
    numbers = [int(v) for v in existing if str(v).isdigit()]
    return str(max(numbers) + 1) if numbers else '1'


# Discovered, never assumed. AML data assets are immutable: we create the next
# version rather than rewrite v1. The storage path follows the version number,
# so a new snapshot can never overwrite the one a deployed model was trained on.
DATA_VERSION = next_version([d.version for d in ml.data.list(name=DATA_ASSET)])
SNAPSHOT_PREFIX = 'telemetry/v' + DATA_VERSION

print('next data asset :', DATA_ASSET + ':' + DATA_VERSION)
print('snapshot path   :', SNAPSHOT_PREFIX)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw = spark.read.format('delta').load(ONELAKE_PATH)

stats = raw.select(
    F.count('*').alias('rows'),
    F.countDistinct('device_id').alias('devices'),
    F.min('timestamp').alias('first_seen'),
    F.max('timestamp').alias('last_seen'),
).collect()[0]

span_min = (stats['last_seen'] - stats['first_seen']).total_seconds() / 60 if stats['rows'] else 0
print('rows       : {:,}'.format(stats['rows']))
print('devices    :', stats['devices'])
print('first_seen :', stats['first_seen'])
print('last_seen  :', stats['last_seen'], ' <- compare with the Kusto max above')
print('span       : {:.0f} min (~{:.1f} h)'.format(span_min, span_min / 60))

if stats['rows'] == 0:
    raise RuntimeError(
        'The Delta mirror is empty. Either mirroring has not caught up yet, or no data '
        'has been ingested since OneLake availability was enabled - the policy was '
        'created with Backfill:false, so older rows may never appear here.')

In [ ]:
# Telemetry arrives ~1 msg/s/device. Collapsing onto a 1-minute grid denoises
# the signal, makes lags expressible as row offsets, and matches the 15-minute
# horizon. It also shrinks the data by a factor of 60.
grid = (
    raw.withColumn('minute', F.date_trunc('minute', F.col('timestamp')))
    .groupBy('device_id', 'minute')
    .agg(*[F.avg(c).alias(c) for c in SENSORS])
)

order = Window.partitionBy('device_id').orderBy('minute')

# Row-based lags jump silently over missing minutes, fabricating relationships
# that never existed. Measure the gaps rather than assume there are none.
gaps = (
    grid.withColumn('prev', F.lag('minute').over(order))
    .withColumn('delta_min', (F.col('minute').cast('long') - F.col('prev').cast('long')) / 60)
    .filter(F.col('delta_min') > 1)
)
gap_count = gaps.count()
print('minutes on the grid : {:,}'.format(grid.count()))
print('gaps > 1 min        :', gap_count)
if gap_count:
    gaps.select('device_id', 'minute', 'delta_min').orderBy(F.desc('delta_min')).show(5, False)

In [ ]:
roll = order.rowsBetween(-(WINDOW_MIN - 1), 0)

feat = grid
for sensor in SENSORS:
    for lag in LAGS_MIN:
        feat = feat.withColumn(sensor + '_lag' + str(lag), F.lag(sensor, lag).over(order))

for sensor in ROLLING_SENSORS:
    # F.stddev is the SAMPLE deviation, matching Kusto's stdevif in
    # features_now(). ddof is not cosmetic: it is a feature the model consumes.
    feat = feat.withColumn(sensor + '_mean' + str(WINDOW_MIN), F.avg(sensor).over(roll))
    feat = feat.withColumn(sensor + '_std' + str(WINDOW_MIN), F.stddev(sensor).over(roll))

feat = (
    feat.withColumn('temperature_slope15',
                    (F.col('temperature_c') - F.col('temperature_c_lag15')) / 15.0)
    .withColumn('hour_of_day', F.hour('minute'))
    .withColumn(TARGET, F.lead('temperature_c', HORIZON_MIN).over(order))
)

# Warm-up rows have no lag history; tail rows have no future label.
dataset = feat.dropna()

feature_columns = [c for c in dataset.columns if c not in ('device_id', 'minute', TARGET)]
print('feature rows    : {:,}'.format(dataset.count()))
print('feature columns :', len(feature_columns))
dataset.select('device_id', 'minute', 'temperature_c', TARGET).orderBy(F.desc('minute')).show(5, False)

In [ ]:
per_device = dataset.groupBy('device_id').count().orderBy('device_id')
per_device.show()

min_rows = per_device.agg(F.min('count')).collect()[0][0] or 0
if min_rows < MIN_ROWS_PER_DEVICE:
    raise RuntimeError(
        'Thinnest device has only {n} usable minutes (floor is {floor}). Replay more '
        'history with --backfill-minutes 1440, wait one 5-minute mirroring window, and '
        're-run. Training on this little data yields a model that reports a fine MAE '
        'and predicts nothing.'.format(n=min_rows, floor=MIN_ROWS_PER_DEVICE))

# If the target barely differs from the current temperature, a model that simply
# echoes its input scores well and forecasts nothing. This is also the
# persistence baseline the pipeline's quality gate compares against.
dataset.select(
    F.mean(F.abs(F.col(TARGET) - F.col('temperature_c'))).alias('mean_abs_change_15min'),
    F.stddev(TARGET).alias('target_std'),
).show(truncate=False)

In [ ]:
import os

# notebookutils.fs.mount authenticates with an Entra token by default - no
# account key, no SAS. The storage account has allowSharedKeyAccess=false, so a
# permission failure here is fixed with an RBAC grant (Storage Blob Data
# Contributor), never with a key.
if any(m.mountPoint == MOUNT_POINT for m in notebookutils.fs.mounts()):
    notebookutils.fs.unmount(MOUNT_POINT)
notebookutils.fs.mount(ABFSS_ROOT, MOUNT_POINT)

local_root = notebookutils.fs.getMountPath(MOUNT_POINT)
output_dir = os.path.join(local_root, 'telemetry', 'v' + DATA_VERSION)
os.makedirs(output_dir, exist_ok=True)
print('mounted', ABFSS_ROOT, '->', local_root)

In [ ]:
# Minute-grain, 4 devices: a single Parquet file is simpler for AML to consume
# and avoids the many-small-files problem.
pdf = dataset.orderBy('device_id', 'minute').toPandas()

output_file = os.path.join(output_dir, 'telemetry_features.parquet')
pdf.to_parquet(output_file, index=False)

print('written:', output_file)
print('shape  : {:,} rows x {} columns'.format(pdf.shape[0], pdf.shape[1]))
print('period :', pdf['minute'].min(), '->', pdf['minute'].max())

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data

# uri_folder, not uri_file: the training component globs the folder, so a later
# snapshot can add partitions without breaking the contract.
asset = ml.data.create_or_update(Data(
    name=DATA_ASSET,
    version=DATA_VERSION,
    type=AssetTypes.URI_FOLDER,
    path='azureml://datastores/' + DATASTORE + '/paths/' + SNAPSHOT_PREFIX,
    description='Minute-grain features snapshotted from the Eventhouse, post regime shift.',
    tags={
        'source': 'fabric-eventhouse',
        'table': 'sensors.' + TABLE_NAME,
        'horizon_min': str(HORIZON_MIN),
        'window_min': str(WINDOW_MIN),
        'rows': str(pdf.shape[0]),
    },
))

print('registered :', asset.name + ':' + asset.version)
print('path       :', asset.path)

### Datastore != data asset

Un **datastore** est un pointeur de connexion : il memorise ou est la donnee et comment
s'y authentifier. Il ne provisionne rien et ne copie rien. Il est cree **une seule fois**,
au provisionnement (`scripts/00_bootstrap.ps1`), et ce notebook n'en cree jamais - il se
contente de le referencer dans le chemin `azureml://datastores/.../paths/...`.

C'est le **data asset** qui porte la version. Chaque entrainement en cree une nouvelle,
avec son propre chemin de stockage. Les versions AML sont **immuables** : une fois
enregistree, une version ne peut plus etre modifiee. La consequence exploitee ici est
concrete - le chemin suit le numero de version (`telemetry/v1`, `telemetry/v2`, ...),
donc un nouveau snapshot ne peut **physiquement pas** ecraser le dossier sur lequel un
modele deja en production a ete entraine.

La cellule suivante affiche les deux cotes en meme temps : le nombre de datastores ne
bouge pas, le nombre de versions augmente. C'est exactement le rapport `1 datastore / N
versions` a montrer en atelier.


In [ ]:
stores = list(ml.datastores.list())
print('DATASTORES ({}) - crees au provisionnement, jamais par un entrainement'.format(len(stores)))
for s in stores:
    marker = '  <-- utilise ici' if s.name == DATASTORE else ''
    print('  {:<28} {}{}'.format(s.name, type(s).__name__, marker))

versions = sorted(ml.data.list(name=DATA_ASSET),
                  key=lambda d: int(d.version) if str(d.version).isdigit() else 0)

print('\nDATA ASSET "{}" ({} versions) - une par entrainement'.format(DATA_ASSET, len(versions)))
for d in versions:
    print('  v{:<3} {:>9} lignes   {}'.format(
        d.version, (d.tags or {}).get('rows', '?'), d.path.rsplit('/paths/', 1)[-1]))

# Chaque version pointe sur son propre dossier. v1 est intacte alors que vN existe:
# le modele deploye peut etre re-entraine a l'identique, aujourd'hui comme dans six mois.
paths = {d.path for d in versions}
assert len(paths) == len(versions), 'Deux versions partagent un chemin: le lineage est casse.'
print('\nOK - {} versions, {} chemins distincts, {} datastore(s).'.format(
    len(versions), len(paths), len(stores)))


## 2. Nouveau modele

Le pipeline enchaine `prep -> train -> evaluate -> register`. L'enregistrement est
**conditionnel** : le modele doit battre une baseline de persistance, pas seulement
afficher une MAE flatteuse.

Le YAML du pipeline reference ses composants par des chemins **relatifs**, et un notebook
Fabric n'a pas le depot. `scripts/42_fabric_p7.ps1` a donc televerse le dossier `aml/`
vers `snapshots/pipeline-source/`. On le recopie sur le disque local du driver plutot que
de travailler directement sur le montage : le SDK cree une archive du dossier de code, et
cette operation se comporte mal a travers un systeme de fichiers FUSE.

In [ ]:
import shutil

SOURCE_ROOT = '/tmp/rti-aml-source'
mounted_source = os.path.join(local_root, PIPELINE_SOURCE_PREFIX)

if not os.path.isdir(mounted_source):
    raise RuntimeError(
        'Pipeline source not found at ' + mounted_source + '. Run scripts/42_fabric_p7.ps1 '
        'from the repository first - it uploads aml/ alongside this notebook.')

if os.path.isdir(SOURCE_ROOT):
    shutil.rmtree(SOURCE_ROOT)
shutil.copytree(mounted_source, SOURCE_ROOT)
os.chdir(SOURCE_ROOT)

print('working directory :', os.getcwd())
print('pipeline          :', os.path.isfile('aml/pipelines/training_pipeline.yml'))
print('components        :', sorted(os.listdir('aml/components')))

In [ ]:
from azure.ai.ml import load_job

versions_before = sorted(int(m.version) for m in ml.models.list(name=MODEL_NAME)
                         if str(m.version).isdigit())
print('model versions before :', versions_before)

job = load_job('aml/pipelines/training_pipeline.yml')
job.inputs.telemetry_features.path = 'azureml:' + DATA_ASSET + ':' + DATA_VERSION
job.inputs.data_asset_tag = DATA_ASSET + ':' + DATA_VERSION
job.display_name = 'rti-temp-forecast-training-v' + DATA_VERSION

submitted = ml.jobs.create_or_update(job)
print('job    :', submitted.name)
print('studio :', submitted.studio_url)

In [ ]:
# Blocks until the pipeline finishes. The compute cluster runs at
# min_instances=0, so the first few minutes are node allocation, not training.
ml.jobs.stream(submitted.name)
print('\nfinal status :', ml.jobs.get(submitted.name).status)

In [ ]:
versions_after = sorted(int(m.version) for m in ml.models.list(name=MODEL_NAME)
                        if str(m.version).isdigit())

if versions_after == versions_before:
    raise RuntimeError(
        'No new model was registered. The quality gate rejected this candidate: it did '
        'not beat the persistence baseline, or it exceeded max_mae. Open the evaluate '
        'step in the studio to see the numbers. This is the gate doing its job.')

# Discovered, never assumed. A retry or a concurrent run shifts the numbering,
# and deploying a guessed version is how you end up serving last week's model
# while narrating this week's.
MODEL_VERSION = str(versions_after[-1])
print('new model :', MODEL_NAME + ':' + MODEL_VERSION)

## 3. Deploiement green, puis 30%

`green` est cree **sans trafic**. Un deploiement neuf peut echouer a demarrer, echouer sa
sonde de sante, ou repondre n'importe quoi : tant qu'il est a 0%, personne ne le voit.

On le teste d'abord a chaud avec `deployment_name='green'`, qui court-circuite la
repartition et vise ce deploiement precisement. **Ensuite** seulement on lui donne du trafic.

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment

# No-code MLflow deployment: no scoring script, no environment. The model's
# MLmodel signature is the contract, which is exactly why blue and green can
# differ by nothing but their model reference.
green = ManagedOnlineDeployment(
    name=GREEN,
    endpoint_name=ENDPOINT,
    model='azureml:' + MODEL_NAME + ':' + MODEL_VERSION,
    instance_type='Standard_DS2_v2',
    instance_count=1,
    tags={'data_asset': DATA_ASSET + ':' + DATA_VERSION},
)

# 8 to 15 minutes: image build, then node allocation. Traffic stays at 0 because
# the split is a property of the ENDPOINT, not of the deployment.
ml.online_deployments.begin_create_or_update(green).result()

print('provisioning :', ml.online_deployments.get(GREEN, ENDPOINT).provisioning_state)
print('traffic      :', ml.online_endpoints.get(ENDPOINT).traffic)

In [ ]:
import tempfile

# Smoke test on REAL current features, not a canned payload. A stale sample file
# would still pass against a broken model whenever the schema happens to match.
live = kql('features_now()')
columns = [c for c in live.columns if c not in ('device_id', 't0')]

# The MLflow 'split' orientation AML expects. Note input_data, not inputs - the
# latter is native MLflow serving and is rejected here.
payload = {
    'input_data': {
        'columns': columns,
        'data': live[columns].astype(float).values.tolist(),
        'index': list(range(len(live))),
    }
}

request_path = os.path.join(tempfile.gettempdir(), 'green_smoke.json')
with open(request_path, 'w') as handle:
    json.dump(payload, handle)

answer = ml.online_endpoints.invoke(
    endpoint_name=ENDPOINT, deployment_name=GREEN, request_file=request_path)

print('devices    :', live['device_id'].tolist())
print('current    :', live['temperature_c'].round(2).tolist())
print('green says :', answer)

In [ ]:
def set_traffic(split):
    endpoint = ml.online_endpoints.get(ENDPOINT)
    endpoint.traffic = split
    ml.online_endpoints.begin_create_or_update(endpoint).result()
    # Read back rather than trust the request: the split is the one thing here
    # that changes what real callers receive.
    return ml.online_endpoints.get(ENDPOINT).traffic


print('traffic :', set_traffic({BLUE: 100 - CANARY_PERCENT, GREEN: CANARY_PERCENT}))
print('\nThe Function was not touched. Same URI, same token, same code.')

In [ ]:
# The loop ticks every 30 s. Give it a minute, then look at what the Function
# recorded. Neither deployment nor model_version is configured anywhere in the
# Function: it reads the deployment from the response header and resolves the
# version against ARM. That is why they follow the split on their own.
display(kql('''
scoring_results
| where scored_at > ago(10m)
| summarize predictions = count(), devices = dcount(device_id), last = max(scored_at)
    by deployment, model_name, model_version
| order by deployment asc
'''))

## 4. Promotion a 100%

`blue` est **conserve** a 0% de trafic. C'est le rollback : le ramener a 100% est une seule
commande, sans reconstruction d'image ni reallocation de noeud.

Il continue a facturer tant qu'il existe. C'est le prix du rollback instantane - un
arbitrage a assumer explicitement, pas un oubli.

In [ ]:
print('traffic :', set_traffic({BLUE: 0, GREEN: 100}))

In [ ]:
display(kql('''
scoring_results
| where scored_at > ago(30m)
| summarize predictions = count() by deployment, model_version, bin(scored_at, 1m)
| order by scored_at desc
| take 20
'''))

## 5. Apres la demo

**Rollback** - quelques secondes, aucune reconstruction :

```python
set_traffic({BLUE: 100, GREEN: 0})
```

**Supprimer blue** - seulement une fois `green` valide sur la duree. Volontairement laisse
en prose : c'est la seule action irreversible de ce notebook.

```python
ml.online_deployments.begin_delete(name=BLUE, endpoint_name=ENDPOINT).result()
```

**Revenir au regime nominal** pour rejouer la demo : relancer le simulateur sans
`--thermal-gain` ni `--ambient-offset`.

**Cout** - l'endpoint facture 24h/24 tant qu'il existe, et chaque deploiement compte pour
un noeud. Deux deploiements = deux noeuds. Voir `scripts/90_teardown.ps1`.